In [1]:
import venti
import numpy as np

In [2]:
print(dir(venti))

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_version', 'models']


In [3]:
from venti.models import plate_motion

In [4]:
# Your GPS data
longitude = np.array([-120.0, -115.0, -110.0, -105.0, -100.0])
latitude = np.array([35.0, 40.0, 45.0, 40.0, 35.0])
velocity_east = np.array([2.1, 1.8, 1.5, 1.2, 0.9])  # mm/year
velocity_north = np.array([0.5, 0.8, 1.1, 1.4, 1.7])  # mm/year
sigma_east = np.array([0.1, 0.15, 0.12, 0.11, 0.13])  # mm/year
sigma_north = np.array([0.12, 0.14, 0.11, 0.13, 0.15])  # mm/year

# Calculate Euler pole
euler_lon, euler_lat, omega, stats = plate_motion.calculate_euler_pole(
    longitude, latitude, velocity_east, velocity_north,
    sigma_east, sigma_north
)

print(f"Euler Pole: {euler_lon:.2f}°, {euler_lat:.2f}°")
print(f"Angular velocity: {omega:.3f} °/Myr")
print(f"RMS: {stats['rms']:.2f} mm/year")


Euler Pole: -136.47°, 57.02°
Angular velocity: 0.039 °/Myr
RMS: 0.59 mm/year


In [5]:
max_sigma, min_sigma, azimuth, sigma_omega = plate_motion.get_euler_pole_uncertainty(
            euler_lon, euler_lat, omega, stats['parameter_covariance'])

In [6]:
# Display results
print(f"\nEULER POLE RESULTS:")
print(f"  Longitude: {euler_lon:.3f}° ± {max_sigma:.3f}°")
print(f"  Latitude:  {euler_lat:.3f}° ± {min_sigma:.3f}°")
print(f"  Rotation:  {omega:.4f} ± {sigma_omega:.4f} °/Myr")

print(f"\nUNCERTAINTY ELLIPSE:")
print(f"  Semi-major axis: {max_sigma:.3f}°")
print(f"  Semi-minor axis: {min_sigma:.3f}°")
print(f"  Ellipse azimuth: {azimuth:.1f}°")
print(f"  Ellipticity: {max_sigma/min_sigma:.2f}")

print(f"\nFIT QUALITY STATISTICS:")
print(f"  RMS:              {stats['rms']:.2f} mm/year")
print(f"  Weighted RMS:     {stats['wrms']:.2f}")
print(f"  Chi-squared:      {stats['chi_squared']:.2f}")
print(f"  Reduced chi-sq:   {stats['reduced_chi_squared']:.2f}")
print(f"  Degrees of freedom: {stats['degrees_of_freedom']}")

print(f"\nINTERPRETATION:")
if stats['reduced_chi_squared'] < 1.5:
    print(f"  Good fit: reduced χ² = {stats['reduced_chi_squared']:.2f} < 1.5")
elif stats['reduced_chi_squared'] < 3.0:
    print(f"  Acceptable fit: reduced χ² = {stats['reduced_chi_squared']:.2f}")
else:
    print(f"  Poor fit: reduced χ² = {stats['reduced_chi_squared']:.2f} > 3.0")

if max_sigma < 5.0:
    print(f"  Good position uncertainty: {max_sigma:.2f}° < 5°")
elif max_sigma < 10.0:
    print(f"  Moderate position uncertainty: {max_sigma:.2f}°")
else:
    print(f"  Large position uncertainty: {max_sigma:.2f}° > 10°")

if sigma_omega / omega < 0.1:
    print(f"  Good rotation rate uncertainty: {sigma_omega/omega:.1%} < 10%")
elif sigma_omega / omega < 0.2:
    print(f"  Moderate rotation rate uncertainty: {sigma_omega/omega:.1%}")
else:
    print(f"  Large rotation rate uncertainty: {sigma_omega/omega:.1%} > 20%")


EULER POLE RESULTS:
  Longitude: -136.471° ± 2.768°
  Latitude:  57.016° ± 0.742°
  Rotation:  0.0390 ± 0.0040 °/Myr

UNCERTAINTY ELLIPSE:
  Semi-major axis: 2.768°
  Semi-minor axis: 0.742°
  Ellipse azimuth: -36.1°
  Ellipticity: 3.73

FIT QUALITY STATISTICS:
  RMS:              0.59 mm/year
  Weighted RMS:     0.41
  Chi-squared:      110.80
  Reduced chi-sq:   3.98
  Degrees of freedom: 7

INTERPRETATION:
  Poor fit: reduced χ² = 3.98 > 3.0
  Good position uncertainty: 2.77° < 5°
  Moderate rotation rate uncertainty: 10.1%


In [7]:
modeled_vel = plate_motion.model_velocities_from_euler_pole(longitude, latitude, -107.2, 50.8, 0.65)
modeled_vel

array([[ 0.02054729, -0.01012073],
       [ 0.01404486, -0.00619798],
       [ 0.00757972, -0.00223026],
       [ 0.01379588,  0.00175313],
       [ 0.02010477,  0.00572545]])

In [8]:
modeled_vel, uncertainties = plate_motion.model_velocities_from_euler_pole(
    longitude, latitude, euler_lon, euler_lat, omega,
    euler_covariance=stats['parameter_covariance']  # From calculate_euler_pole()
)
modeled_vel, uncertainties

(array([[0.00169192, 0.00066871],
        [0.00138587, 0.00086304],
        [0.00108976, 0.00105069],
        [0.00150316, 0.00123095],
        [0.00190039, 0.00140193]]),
 array([[6.37745274e-05, 8.30736885e-05],
        [5.39155408e-05, 6.36335797e-05],
        [7.38592523e-05, 5.75111709e-05],
        [5.38203543e-05, 6.85602724e-05],
        [6.43157007e-05, 9.04695024e-05]]))

In [9]:
itrf14_pmm = plate_motion.json_to_dataframe(plate_motion.load_itrf_json(date=2014))
itrf20_pmm = plate_motion.json_to_dataframe(plate_motion.load_itrf_json(date=2020))

In [11]:
import pandas as pd
itrf14_gps = pd.read_csv(plate_motion.ITRF14_DATA)
itrf20_gps = pd.read_csv(plate_motion.ITRF20_DATA)

In [18]:
pacific_data = itrf14_gps[itrf14_gps['Plate'] == 'Pacific']

# Calculate Euler pole
longitude = pacific_data['Longitude_deg'].values
latitude = pacific_data['Latitude_deg'].values
ve = pacific_data['Ve_mm_yr'].values
vn = pacific_data['Vn_mm_yr'].values
se = pacific_data['Se_mm_yr'].values
sn = pacific_data['Sn_mm_yr'].values

euler_lon, euler_lat, omega, stats = plate_motion.calculate_euler_pole(
    longitude, latitude, ve, vn, se, sn, correlation_coefficient=0.35, 
)

# Calculate uncertainties
max_sigma, min_sigma, azimuth, sigma_omega = plate_motion.get_euler_pole_uncertainty(
    euler_lon, euler_lat, omega, stats['parameter_covariance']
)

print(f"\nCalculated Pacific Plate Euler Pole:")
print(f"  Longitude: {euler_lon:.3f}°")
print(f"  Latitude:  {euler_lat:.3f}°")
print(f"  Angular velocity: {omega:.4f} °/Myr")
print(f"  RMS: {stats['rms']:.2f} mm/year")

plate_motion.euler_pole_to_rotation_rate(euler_lon, euler_lat, np.rad2deg(omega)*1e6)


Calculated Pacific Plate Euler Pole:
  Longitude: 111.197°
  Latitude:  -62.524°
  Angular velocity: 0.6796 °/Myr
  RMS: 0.52 mm/year


(np.float64(-0.1133680479026246),
 np.float64(0.2923201917370568),
 np.float64(-0.6029138285387291))

In [19]:
itrf14_pmm[itrf14_pmm.name == 'Pacific Plate'] 

,plate,name,omega_x,omega_y,omega_z
8,PCFC,Pacific Plate,-0.1135,0.2907,-0.6025


In [30]:
pacific_data = itrf20_gps[itrf20_gps['Plate'] == 'Pacific']

# Calculate Euler pole
longitude = pacific_data['Longitude_deg'].values
latitude = pacific_data['Latitude_deg'].values
ve = pacific_data['Ve_mm_yr'].values
vn = pacific_data['Vn_mm_yr'].values
se = pacific_data['Se_mm_yr'].values
sn = pacific_data['Sn_mm_yr'].values

euler_lon, euler_lat, omega, stats = plate_motion.calculate_euler_pole(
    longitude, latitude, ve, vn, se, sn, correlation_coefficient=0, 
)

# Calculate uncertainties
max_sigma, min_sigma, azimuth, sigma_omega = plate_motion.get_euler_pole_uncertainty(
    euler_lon, euler_lat, omega, stats['parameter_covariance']
)

print(f"\nCalculated Pacific Plate Euler Pole:")
print(f"  Longitude: {euler_lon:.3f}°")
print(f"  Latitude:  {euler_lat:.3f}°")
print(f"  Angular velocity: {omega:.4f} °/Myr")
print(f"  RMS: {stats['rms']:.2f} mm/year")

plate_motion.euler_pole_to_rotation_rate(euler_lon, euler_lat, np.rad2deg(omega)*1e6)


Calculated Pacific Plate Euler Pole:
  Longitude: 111.374°
  Latitude:  -62.590°
  Angular velocity: 0.6779 °/Myr
  RMS: 0.53 mm/year


(np.float64(-0.11372905248471399),
 np.float64(0.2905938206180149),
 np.float64(-0.6017494583276538))

In [31]:
itrf20_pmm[itrf20_pmm.name == 'Pacific Plate'] 

,plate,name,omega_x,omega_y,omega_z
10,PCFC,Pacific Plate,-0.1122,0.2836,-0.5984
